# Kicker Bundesliga Predicted Lineups

This notebook prompts for a Bundesliga matchday in the fixed 2026-27 season, collects every Kicker fixture's lineup page through one undetected Chrome session, and writes a lossless predicted-lineup snapshot.

Kicker's displayed lineup order is preserved. The first four paragraphs in each `kick__lineup-text` block are interpreted as starting XI, coach, bench, and unavailable players respectively; their original HTML and text are retained alongside the parsed fields.

## 1. Resolve project and output paths

In [1]:
import sys
from pathlib import Path


def locate_project_root() -> Path:
    """Find the repository when run from JupyterLab or VS Code."""
    starts = []
    vscode_notebook = globals().get('__vsc_ipynb_file__')
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())
    checked = set()
    for start in starts:
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError(
        'Could not locate project_paths.py. Start Jupyter from the Kickbase project root.'
    )


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import KICKER_PREDICTED_LINEUPS_DIR, ensure_directory

output_directory = ensure_directory(KICKER_PREDICTED_LINEUPS_DIR)
print(f'Kicker lineup output directory: {output_directory}')

Kicker lineup output directory: C:\kickbase project\outputs\kicker\predicted_lineups


## 2. Imports, source configuration, and matchday input

In [2]:
import json
import re
from datetime import datetime
from typing import Any
from urllib.parse import urljoin

try:
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        'Required packages are missing. Install them with: '
        '%pip install beautifulsoup4 undetected-chromedriver selenium'
    ) from exc

SOURCE_NAME = 'Kicker'
SEASON = '2026-27'
SCHEDULE_URL_TEMPLATE = 'https://www.kicker.de/bundesliga/spieltag/{season}/{matchday}'
CHROME_MAJOR_VERSION = None  # Let undetected-chromedriver auto-detect Chrome.
PAGE_LOAD_TIMEOUT_SECONDS = 45
WAIT_TIMEOUT_SECONDS = 30
MIN_MATCHDAY = 1
MAX_MATCHDAY = 34
GAME_DATE_GROUP_SELECTOR = 'div.kick__v100-gameList.kick__module-margin'
GAME_ROW_SELECTOR = 'div.kick__v100-gameList__gameRow'
SCHEMA_LINK_SELECTOR = (
    'a.kick__v100-gameList__gameRow__stateCell__indicator'
    '.kick__v100-gameList__gameRow__stateCell__indicator--schema'
)
LINEUP_BLOCK_SELECTOR = 'div.kick__lineup-text'
LINEUP_EXPLANATION_SELECTOR = '.kick__lineup-explanation'
CHALLENGE_MARKERS = (
    "verify that you're not a robot",
    'verify you are human',
    'verify that you are human',
    'javascript is disabled',
    'access denied',
)


def prompt_matchday() -> int:
    raw_matchday = input(f'Bundesliga matchday ({MIN_MATCHDAY}-{MAX_MATCHDAY}): ').strip()
    try:
        matchday = int(raw_matchday)
    except ValueError as exc:
        raise ValueError('Matchday must be a whole number.') from exc
    if not MIN_MATCHDAY <= matchday <= MAX_MATCHDAY:
        raise ValueError(f'Matchday must be between {MIN_MATCHDAY} and {MAX_MATCHDAY}.')
    return matchday


matchday = prompt_matchday()
source_url = SCHEDULE_URL_TEMPLATE.format(season=SEASON, matchday=matchday)
print(f'Kicker source URL: {source_url}')

Bundesliga matchday (1-34):  2


Kicker source URL: https://www.kicker.de/bundesliga/spieltag/2026-27/2


## 3. Parse Kicker schedule rows and lineup blocks

In [3]:
def clean_text(value: Any) -> str | None:
    if value is None:
        return None
    cleaned = re.sub(r'\s+', ' ', str(value).replace('\xa0', ' ')).strip()
    return cleaned or None


def linked_player(anchor: Any, fixture_url: str, formation_row: int | None = None, slot_index: int | None = None) -> dict[str, Any]:
    displayed_name = clean_text(anchor.get_text(' ', strip=True))
    href = clean_text(anchor.get('href'))
    if displayed_name is None or href is None:
        raise ValueError('A Kicker player link is missing its displayed name or URL.')
    return {
        'full_name': None,
        'displayed_name': displayed_name,
        'position': None,
        'injury_status': None,
        'player_id': None,
        'player_url': urljoin(fixture_url, href),
        'formation_row': formation_row,
        'slot_index': slot_index,
        'starting_probability_rank': 1,
    }


def parse_starting_xi(paragraph: Any | None, fixture_url: str) -> dict[str, Any]:
    raw_text = clean_text(paragraph.get_text(' ', strip=True)) if paragraph else None
    rows: list[list[dict[str, Any]]] = [[]]
    if paragraph is not None:
        for child in paragraph.children:
            if getattr(child, 'name', None) == 'a':
                rows[-1].append(linked_player(child, fixture_url))
            elif getattr(child, 'name', None) is None and re.search(r'\s[-–]\s', str(child)):
                rows.append([])
    rows = [row for row in rows if row]
    for formation_row, row in enumerate(rows, start=1):
        for slot_index, player in enumerate(row, start=1):
            player['formation_row'] = formation_row
            player['slot_index'] = slot_index
    line_counts = [len(row) for row in rows]
    outfield_counts = line_counts[1:] if line_counts[:1] == [1] else line_counts
    formation = '-'.join(str(count) for count in outfield_counts) or None
    return {
        'raw_text': raw_text,
        'formation': formation,
        'formation_lines': outfield_counts,
        'formation_row_counts': line_counts,
        'players': [player for row in rows for player in row],
    }


def parse_linked_list(paragraph: Any | None, fixture_url: str) -> list[dict[str, Any]]:
    if paragraph is None:
        return []
    return [linked_player(anchor, fixture_url) for anchor in paragraph.select('a[href]')]


def parse_unavailable(paragraph: Any | None, fixture_url: str) -> dict[str, Any]:
    raw_text = clean_text(paragraph.get_text(' ', strip=True)) if paragraph else None
    players = []
    for anchor in paragraph.select('a[href]') if paragraph is not None else []:
        suffix = clean_text(anchor.next_sibling) or ''
        reason_match = re.search(r'\(([^()]*)\)', suffix)
        player = linked_player(anchor, fixture_url)
        players.append({
            'raw_text': clean_text(f"{player['displayed_name']}{suffix}"),
            'name': player['displayed_name'],
            'reason': clean_text(reason_match.group(1)) if reason_match else None,
            'player_url': player['player_url'],
        })
    return {'raw_text': raw_text, 'players': players}


def parse_lineup_block(block: Any, lineup_index: int, fixture_url: str) -> dict[str, Any]:
    paragraphs = []
    for paragraph in block.find_all('p'):
        paragraphs.append({
            'index': len(paragraphs) + 1,
            'outer_html': str(paragraph),
            'text': clean_text(paragraph.get_text(' ', strip=True)),
        })
    paragraph_tags = block.find_all('p')
    starting_paragraph = paragraph_tags[0] if paragraph_tags else None
    coach_paragraph = paragraph_tags[1] if len(paragraph_tags) > 1 else None
    bench_paragraph = paragraph_tags[2] if len(paragraph_tags) > 2 else None
    unavailable_paragraph = paragraph_tags[3] if len(paragraph_tags) > 3 else None
    coach_players = parse_linked_list(coach_paragraph, fixture_url)
    return {
        'lineup_index': lineup_index,
        'raw_html': str(block),
        'paragraphs': paragraphs,
        'starting_xi': parse_starting_xi(starting_paragraph, fixture_url),
        'coach': {
            'raw_text': clean_text(coach_paragraph.get_text(' ', strip=True)) if coach_paragraph else None,
            'name': coach_players[0]['displayed_name'] if coach_players else None,
            'coach_url': coach_players[0]['player_url'] if coach_players else None,
        },
        'bench': {
            'raw_text': clean_text(bench_paragraph.get_text(' ', strip=True)) if bench_paragraph else None,
            'players': parse_linked_list(bench_paragraph, fixture_url),
        },
        'unavailable': parse_unavailable(unavailable_paragraph, fixture_url),
    }


def extract_fixture_links(schedule_html: str, schedule_url: str) -> list[dict[str, str]]:
    soup = BeautifulSoup(schedule_html, 'html.parser')
    date_groups = soup.select(GAME_DATE_GROUP_SELECTOR)
    if not date_groups:
        raise ValueError('No Kicker match-date groups were found on the schedule page.')
    fixtures = []
    seen_urls = set()
    for date_group in date_groups:
        for game_row in date_group.select(GAME_ROW_SELECTOR):
            link = game_row.select_one(SCHEMA_LINK_SELECTOR)
            href = clean_text(link.get('href') if link else None)
            if href is None:
                continue
            fixture_url = urljoin(schedule_url, href)
            if fixture_url in seen_urls:
                continue
            seen_urls.add(fixture_url)
            fixtures.append({
                'fixture_url': fixture_url,
                'schedule_row_text': clean_text(game_row.get_text(' ', strip=True)) or '',
            })
    if not fixtures:
        raise ValueError('No Kicker lineup schema links were found on the schedule page.')
    return fixtures


def extract_team_name(block: Any) -> str | None:
    logo = block.select_one('.kick__lineup-text__headline img[alt]')
    if logo is not None:
        return clean_text(logo.get('alt'))
    headline = block.select_one('.kick__lineup-text__headline strong')
    return clean_text(headline.get_text(' ', strip=True)) if headline else None


def extract_match_time(soup: Any, schedule_row_text: str) -> str:
    time_element = soup.select_one('time[datetime]') or soup.select_one('time')
    if time_element is not None:
        return clean_text(time_element.get('datetime')) or clean_text(
            time_element.get_text(' ', strip=True)
        ) or schedule_row_text
    schedule_time = re.search(r'\b(?:Mo|Di|Mi|Do|Fr|Sa|So)\.\s*\d{1,2}:\d{2}\b', schedule_row_text)
    return clean_text(schedule_time.group(0)) if schedule_time else schedule_row_text


def make_kicker_players(starting_xi: dict[str, Any]) -> list[dict[str, Any]]:
    return [dict(player) for player in starting_xi['players']]


def make_kicker_team(lineup: dict[str, Any], team_name: str | None, side: str) -> dict[str, Any]:
    return {
        'team_name': team_name,
        'side': side,
        'lineup_status': 'Predicted Lineup',
        'players': make_kicker_players(lineup['starting_xi']),
        'kicker_details': {
            key: lineup[key]
            for key in ('lineup_index', 'raw_html', 'paragraphs', 'starting_xi', 'coach', 'bench', 'unavailable')
        },
    }


def parse_fixture_page(fixture_html: str, fixture: dict[str, str]) -> dict[str, Any] | None:
    soup = BeautifulSoup(fixture_html, 'html.parser')
    blocks = soup.select(LINEUP_BLOCK_SELECTOR)
    if not blocks:
        return None
    if len(blocks) != 2:
        raise ValueError(f'Expected two Kicker lineup blocks, found {len(blocks)}.')
    home_lineup, away_lineup = [
        parse_lineup_block(block, index, fixture['fixture_url'])
        for index, block in enumerate(blocks, start=1)
    ]
    return {
        'match_time': extract_match_time(soup, fixture['schedule_row_text']),
        'home': make_kicker_team(home_lineup, extract_team_name(blocks[0]), 'home'),
        'away': make_kicker_team(away_lineup, extract_team_name(blocks[1]), 'away'),
        'kicker_details': {
            **fixture,
            'lineup_block_count': len(blocks),
            'explanations': [
                {
                    'outer_html': str(explanation),
                    'text': clean_text(explanation.get_text(' ', strip=True)),
                }
                for explanation in soup.select(LINEUP_EXPLANATION_SELECTOR)
            ],
        },
    }


def page_contains_challenge(page_html: str) -> bool:
    visible_text = BeautifulSoup(page_html, 'html.parser').get_text(' ', strip=True).casefold()
    return any(marker in visible_text for marker in CHALLENGE_MARKERS)


def wait_for_required_elements(current_driver: Any, selector: str, page_label: str) -> None:
    if page_contains_challenge(current_driver.page_source):
        print(
            f'Kicker is requesting a browser verification for the {page_label}. '
            'Complete it yourself in the Chrome window, then return here.'
        )
        input('Press Enter after the Kicker page has loaded normally: ')
    try:
        WebDriverWait(current_driver, WAIT_TIMEOUT_SECONDS).until(
            lambda active_driver: active_driver.find_elements(By.CSS_SELECTOR, selector)
        )
    except TimeoutException as exc:
        if page_contains_challenge(current_driver.page_source):
            raise RuntimeError(
                f'Kicker verification is still active for the {page_label}. '
                'Complete it manually, then rerun this cell.'
            ) from exc
        raise


def wait_for_document(current_driver: Any) -> bool:
    return current_driver.execute_script('return document.readyState') == 'complete'


## 4. Run local parser checks

These checks cover date-group iteration, deduplication, the four documented paragraph roles, and a shortened block with optional paragraphs omitted.

In [4]:
schedule_fixture_html = '''
<div class="kick__v100-gameList kick__module-margin">
  <div class="kick__v100-gameList__gameRow"><a class="kick__v100-gameList__gameRow__stateCell__indicator kick__v100-gameList__gameRow__stateCell__indicator--schema" href="/fixture-a"></a>Fixture A</div>
</div>
<div class="kick__v100-gameList kick__module-margin">
  <div class="kick__v100-gameList__gameRow"><a class="kick__v100-gameList__gameRow__stateCell__indicator kick__v100-gameList__gameRow__stateCell__indicator--schema" href="/fixture-b"></a>Fixture B</div>
  <div class="kick__v100-gameList__gameRow"><a class="kick__v100-gameList__gameRow__stateCell__indicator kick__v100-gameList__gameRow__stateCell__indicator--schema" href="/fixture-a"></a>Fixture A duplicate</div>
</div>
'''
fixture_links = extract_fixture_links(schedule_fixture_html, 'https://www.kicker.de/bundesliga/spieltag/2026-27/1')
assert [item['fixture_url'] for item in fixture_links] == [
    'https://www.kicker.de/fixture-a',
    'https://www.kicker.de/fixture-b',
]
lineup_fixture_html = '''
<time datetime="2026-08-28T20:30+02:00"></time>
<div class="kick__lineup-text">
  <div class="kick__lineup-text__headline"><img alt="Home FC"/></div>
  <p><strong>Voraussichtliche Aufstellung</strong><br/><a href="/goalkeeper/spieler">Goalkeeper</a> - <a href="/right-back/spieler">Right Back</a>, <a href="/centre-back/spieler">Centre Back</a> - <a href="/midfielder/spieler">Midfielder</a> - <a href="/forward/spieler">Forward</a></p>
  <p><strong>Trainer</strong><br/><a href="/sample-coach/trainer">Sample Coach</a></p>
  <p><strong>Reservebank</strong><br/><a href="/substitute-one/spieler">Substitute One</a>, <a href="/substitute-two/spieler">Substitute Two</a></p>
  <p><strong>Es fehlen</strong><br/><a href="/injured-player/spieler">Injured Player</a> (muscle injury), <a href="/rested-player/spieler">Rested Player</a> (rested)</p>
</div>
<div class="kick__lineup-text"><div class="kick__lineup-text__headline"><img alt="Away FC"/></div><p><strong>Voraussichtliche Aufstellung</strong><br/><a href="/away-goalkeeper/spieler">Away Goalkeeper</a></p></div>
<p class="kick__lineup-explanation">Explanation <strong>kept verbatim</strong>.</p>
'''
parsed_fixture = parse_fixture_page(lineup_fixture_html, fixture_links[0])
assert parsed_fixture is not None
assert parsed_fixture['match_time'] == '2026-08-28T20:30+02:00'
assert parsed_fixture['home']['team_name'] == 'Home FC'
assert parsed_fixture['away']['team_name'] == 'Away FC'
assert parsed_fixture['home']['kicker_details']['starting_xi']['formation_lines'] == [2, 1, 1]
assert [player['displayed_name'] for player in parsed_fixture['home']['players']] == ['Goalkeeper', 'Right Back', 'Centre Back', 'Midfielder', 'Forward']
assert parsed_fixture['home']['players'][0]['formation_row'] == 1
assert parsed_fixture['home']['players'][1]['slot_index'] == 1
assert parsed_fixture['home']['players'][0]['player_url'] == 'https://www.kicker.de/goalkeeper/spieler'
assert parsed_fixture['home']['kicker_details']['coach']['name'] == 'Sample Coach'
assert [player['displayed_name'] for player in parsed_fixture['home']['kicker_details']['bench']['players']] == ['Substitute One', 'Substitute Two']
assert parsed_fixture['home']['kicker_details']['unavailable']['players'][0]['reason'] == 'muscle injury'
assert parsed_fixture['away']['players'][0]['formation_row'] == 1
assert '<strong>kept verbatim</strong>' in parsed_fixture['kicker_details']['explanations'][0]['outer_html']
assert not page_contains_challenge('<script>const captchaProvider = "example";</script>')
assert page_contains_challenge('<main>Verify that you are human</main>')
print('Local Kicker parser checks passed.')

Local Kicker parser checks passed.


## 5. Load the Kicker schedule and every fixture

A fixture without lineup blocks is retained in `skipped_fixtures` with its reason. A challenge or unavailable schedule page stops the run before any output file is written.

In [5]:
scrape_started_at = datetime.now().astimezone()
scrape_finished_at = None
driver = None
fixtures = []
matches = []
skipped_fixtures = []

try:
    driver = uc.Chrome()
    driver.set_window_size(1920, 1080)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
    wait = WebDriverWait(driver, WAIT_TIMEOUT_SECONDS)

    driver.get(source_url)
    wait.until(wait_for_document)
    wait_for_required_elements(driver, GAME_DATE_GROUP_SELECTOR, 'schedule page')
    fixtures = extract_fixture_links(driver.page_source, source_url)
    print(f'Found {len(fixtures)} unique Kicker fixture links.')

    for fixture_number, fixture in enumerate(fixtures, start=1):
        try:
            driver.get(fixture['fixture_url'])
            wait.until(wait_for_document)
            wait_for_required_elements(driver, LINEUP_BLOCK_SELECTOR, 'fixture page')
            parsed_fixture = parse_fixture_page(driver.page_source, fixture)
            if parsed_fixture is None:
                skipped_fixtures.append({**fixture, 'reason': 'No Kicker lineup blocks found.'})
            else:
                matches.append(parsed_fixture)
                print(
                    f"{fixture_number:>2}. Parsed {parsed_fixture['home']['team_name']} vs "
                    f"{parsed_fixture['away']['team_name']}: {fixture['fixture_url']}"
                )
        except TimeoutException:
            latest_html = driver.page_source if driver is not None else ''
            reason = (
                'Kicker presented a bot/challenge page.'
                if page_contains_challenge(latest_html)
                else 'Timed out waiting for Kicker lineup blocks.'
            )
            skipped_fixtures.append({**fixture, 'reason': reason})
        except (RuntimeError, ValueError, WebDriverException) as exc:
            skipped_fixtures.append({**fixture, 'reason': str(exc)})
except TimeoutException as exc:
    latest_html = driver.page_source if driver is not None else ''
    if page_contains_challenge(latest_html):
        raise RuntimeError('Kicker presented a bot/challenge page for the schedule URL.') from exc
    raise RuntimeError('Timed out while loading the Kicker schedule page.') from exc
except WebDriverException as exc:
    raise RuntimeError(
        'Could not load Kicker through undetected Chrome. Confirm that compatible Chrome is installed.'
    ) from exc
finally:
    scrape_finished_at = datetime.now().astimezone()
    if driver is not None:
        try:
            driver.quit()
            driver.quit = lambda: None
            print('Chrome driver closed.')
        except Exception as shutdown_error:
            print(f'Chrome shutdown warning: {type(shutdown_error).__name__}: {shutdown_error}')
        finally:
            driver = None

if not fixtures:
    raise RuntimeError('Kicker did not provide any fixture links; no JSON snapshot was written.')
print(f'Extracted fixtures: {len(matches)}; skipped fixtures: {len(skipped_fixtures)}')

Found 9 unique Kicker fixture links.
 1. Parsed VfB Stuttgart vs 1. FC Köln: https://www.kicker.de/stuttgart-gegen-koeln-2026-bundesliga-5226818/aufstellung
 2. Parsed TSG Hoffenheim vs Borussia Dortmund: https://www.kicker.de/hoffenheim-gegen-dortmund-2026-bundesliga-5226819/aufstellung
 3. Parsed Bayer 04 Leverkusen vs 1. FC Union Berlin: https://www.kicker.de/leverkusen-gegen-union-2026-bundesliga-5226820/aufstellung
 4. Parsed Bor. Mönchengladbach vs SV Elversberg: https://www.kicker.de/mgladbach-gegen-elversberg-2026-bundesliga-5226822/aufstellung
 5. Parsed Werder Bremen vs RB Leipzig: https://www.kicker.de/bremen-gegen-leipzig-2026-bundesliga-5226824/aufstellung
 6. Parsed SC Paderborn 07 vs SC Freiburg: https://www.kicker.de/paderborn-gegen-freiburg-2026-bundesliga-5226826/aufstellung
 7. Parsed FC Schalke 04 vs Bayern München: https://www.kicker.de/schalke-gegen-bayern-2026-bundesliga-5226825/aufstellung
 8. Parsed Hamburger SV vs 1. FSV Mainz 05: https://www.kicker.de/hsv-geg

## 6. Validate and write the timestamped JSON snapshot

In [6]:
team_count = sum(2 for _ in matches)
player_count = sum(
    len(match[side]['players'])
    for match in matches
    for side in ('home', 'away')
)
lineup_block_count = sum(match['kicker_details']['lineup_block_count'] for match in matches)
output_data = {
    'metadata': {
        'source': SOURCE_NAME,
        'source_url': source_url,
        'season': SEASON,
        'matchday': matchday,
        'captured_at': scrape_started_at.isoformat(timespec='seconds'),
        'capture_finished_at': scrape_finished_at.isoformat(timespec='seconds'),
        'fixture_link_count': len(fixtures),
        'match_count': len(matches),
        'team_count': team_count,
        'player_count': player_count,
        'lineup_block_count': lineup_block_count,
        'skipped_fixture_count': len(skipped_fixtures),
        'skipped_fixtures': skipped_fixtures,
    },
    'matches': matches,
}

filename_timestamp = scrape_started_at.strftime('%Y%m%d_%H%M%S')
output_path = output_directory / f'kicker_bundesliga_lineups_{filename_timestamp}.json'
temporary_output_path = output_path.with_suffix('.json.tmp')
try:
    with temporary_output_path.open('w', encoding='utf-8', newline='\n') as file:
        json.dump(output_data, file, ensure_ascii=False, indent=2)
        file.write('\n')
    temporary_output_path.replace(output_path)
except OSError as exc:
    raise OSError(f'Could not write Kicker snapshot {output_path}: {exc}') from exc
finally:
    temporary_output_path.unlink(missing_ok=True)

with output_path.open('r', encoding='utf-8') as file:
    validated_output = json.load(file)
if validated_output['metadata']['matchday'] != matchday:
    raise ValueError('Saved Kicker JSON has an unexpected matchday.')
if len(validated_output['matches']) != len(matches):
    raise ValueError('Saved Kicker JSON has an unexpected match count.')
print(f'Saved Kicker lineup snapshot: {output_path.resolve()}')

Saved Kicker lineup snapshot: C:\kickbase project\outputs\kicker\predicted_lineups\kicker_bundesliga_lineups_20260904_105209.json


## 7. Report captured fixtures

In [7]:
for match_number, match in enumerate(matches, start=1):
    print(
        f"{match_number:>2}. {match['match_time']} | "
        f"{match['home']['team_name']} vs {match['away']['team_name']} | "
        f"{match['kicker_details']['fixture_url']}"
    )

if skipped_fixtures:
    print('\nSkipped fixtures:')
    for fixture in skipped_fixtures:
        print(f"- {fixture['fixture_url']}: {fixture['reason']}")

print(f'\nJSON output: {output_path.resolve()}')

 1. VfB Stuttgart Stuttgart 20:30 heute 1. FC Köln Köln Vorschau | VfB Stuttgart vs 1. FC Köln | https://www.kicker.de/stuttgart-gegen-koeln-2026-bundesliga-5226818/aufstellung
 2. Sa. 15:30 | TSG Hoffenheim vs Borussia Dortmund | https://www.kicker.de/hoffenheim-gegen-dortmund-2026-bundesliga-5226819/aufstellung
 3. Sa. 15:30 | Bayer 04 Leverkusen vs 1. FC Union Berlin | https://www.kicker.de/leverkusen-gegen-union-2026-bundesliga-5226820/aufstellung
 4. Sa. 15:30 | Bor. Mönchengladbach vs SV Elversberg | https://www.kicker.de/mgladbach-gegen-elversberg-2026-bundesliga-5226822/aufstellung
 5. Sa. 15:30 | Werder Bremen vs RB Leipzig | https://www.kicker.de/bremen-gegen-leipzig-2026-bundesliga-5226824/aufstellung
 6. Sa. 15:30 | SC Paderborn 07 vs SC Freiburg | https://www.kicker.de/paderborn-gegen-freiburg-2026-bundesliga-5226826/aufstellung
 7. Sa. 18:30 | FC Schalke 04 vs Bayern München | https://www.kicker.de/schalke-gegen-bayern-2026-bundesliga-5226825/aufstellung
 8. So. 15:30 | H